# Does cortical growth help once newborn cells mature?

Built from commit `f8113fa`. A small population, a long continual stream of near-identical sentence pairs,
and five arms: frozen, tuning, v17's instant growth, growth with maturation, and a population that starts at
the grown size. See `PREREGISTRATION_maturation.md`.

In [ ]:
import os, sys, subprocess
W = "/kaggle/working/brain"
for d in ("brain", "tests"):
    os.makedirs(os.path.join(W, d), exist_ok=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
%%writefile /kaggle/working/brain/brain/__init__.py
"""A conversational brain: lexical cortex, prefrontal working memory, hippocampal map and episodic store, a
basal-ganglia gate that learns which of them answers, and sleep. No pretrained weights, no gradients."""

In [ ]:
%%writefile /kaggle/working/brain/brain/maturation.py
"""Does cortical growth help once new cells are given time to mature and the task outgrows the population?

v17 found growth hurt (critical 0.760 separation vs tune-only 0.808). Two criticisms of that test, both fair:

  maturation   a v17 newborn joined the code at full strength the moment it was wired, and could be culled
               after 12 exposures. Adult-born neurons take weeks to mature: they are hyperexcitable, broadly
               tuned and weakly connected to their targets at first, unusually plastic for a window, and only
               then selective and integrated (Ge et al. 2007; Marin-Burgin et al. 2012; Aimone et al. 2011).
  capacity     tuning alone reached 0.81 on v17's curriculum, so the population it had was already enough and
               growth had nothing to add.

This experiment fixes both. A small population meets a long *continual* stream of thousands of near-identical
sentence pairs, so it runs out of room; newborn cells go through a maturation period - silent to the code at
first, then integrated gradually, with extra plasticity while young, and immune to culling until mature. And it
adds the control v17 lacked: a population that simply starts at the size the growing one ends at. If growth only
beats tuning, the benefit may be "more cells", not "growing them".

Everything is vectorised torch, on the GPU when there is one.

The population (same cell as brain/dendrite.py): each cell has `branches` dendritic branches of `per` synapses;
branch drive s is summed, passed through s^2/(theta+s) (supralinear, saturating), branches add at the soma,
divisive gain control, k winners fire. Tuning = homeostasis (bias toward the target rate) + branch-specific
adaptation + instar Hebbian plasticity (each winner's synapses move toward the input it saw).

Inputs are position-bound word codes, as Wernicke hears them: every word has a code and every word-at-position
has a code, and a sentence is the union - so "Rex is older than Max" and "Max is older than Rex" share every
word code and differ only in conjunctions.
"""
import json
import math
import random
import time
import zlib

import torch

# ------------------------------------------------------------------------------------------ the sentences
SYL_C = "bdfgklmnprstvz"
SYL_V = "aeiou"


def names(count, seed):
    """Invented two-syllable names, and for each a near-twin one letter away (Cara / Cora)."""
    rng = random.Random(seed)
    out, seen = [], set()
    while len(out) < count:
        n = "".join(rng.choice(SYL_C) + rng.choice(SYL_V) for _ in range(2)).capitalize()
        i = rng.randrange(1, len(n))
        twin = n[:i] + rng.choice(SYL_V if n[i] in SYL_V else SYL_C) + n[i + 1:]
        if n != twin and n not in seen and twin not in seen:
            seen |= {n, twin}
            out.append((n, twin))
    return out


SCALES = ["older", "taller", "faster", "heavier", "richer", "louder", "kinder", "braver"]
PLACES = ["park", "pack", "cafe", "cove", "station", "stables", "harbour", "arbour", "market", "marker", "mill",
          "hill", "bridge", "ridge", "tower", "power"]


def stream_pairs(count, seed):
    """`count` pairs of near-identical sentences that mean different things - v17's hard grades, at scale."""
    rng = random.Random(seed)
    people = names(max(64, count // 3), seed)
    flat = [n for pair in people for n in pair]
    out = []
    for _ in range(count):
        kind = rng.choice(["swap", "twin", "negation", "tense", "place", "direction", "scale"])
        a, b = rng.sample(flat, 2)
        w = rng.choice(SCALES)
        if kind == "swap":
            out.append((f"{a} is {w} than {b}", f"{b} is {w} than {a}"))
        elif kind == "twin":
            one, two = rng.choice(people)
            out.append((f"{one} is {w} than {b}", f"{two} is {w} than {b}"))
        elif kind == "negation":
            out.append((f"{a} is {w} than {b}", f"{a} is not {w} than {b}"))
        elif kind == "tense":
            out.append((f"{a} is {w} than {b}", f"{a} was {w} than {b}"))
        elif kind == "place":
            i = rng.randrange(0, len(PLACES), 2)
            out.append((f"{a} lives north of the {PLACES[i]}", f"{a} lives north of the {PLACES[i + 1]}"))
        elif kind == "direction":
            p = rng.choice(PLACES)
            out.append((f"{a} lives north of the {p}", f"{a} lives south of the {p}"))
        else:
            w2 = rng.choice([s for s in SCALES if s != w])
            out.append((f"{a} is {w} than {b}", f"{a} is {w2} than {b}"))
    return out


def encode(sentences, d_in=4096, k_word=8, k_conj=8):
    """Position-bound codes: each word drives k_word fibres, each word-at-position k_conj more."""
    x = torch.zeros(len(sentences), d_in)
    for r, s in enumerate(sentences):
        for i, w in enumerate(s.lower().split()):
            for key, k in ((w, k_word), (f"{w}@{i}", k_conj)):
                g = torch.Generator().manual_seed(zlib.crc32(key.encode()))
                x[r, torch.randint(d_in, (k,), generator=g)] = 1.0
    return x


# ------------------------------------------------------------------------------------------ the population
class Population:
    def __init__(self, d_in, n, k, branches=4, per=6, seed=0, device="cpu", theta=1.0, divisive=0.5,
                 homeostasis=0.0, adapt=0.0, plastic=0.0):
        self.dev = device
        g = torch.Generator().manual_seed(seed)
        self.d_in, self.k, self.B, self.S = d_in, k, branches, per
        self.theta, self.divisive = theta, divisive
        self.homeostasis, self.adapt, self.plastic = homeostasis, adapt, plastic
        self.idx = torch.randint(d_in, (n, branches, per), generator=g).to(device)
        self.w = (0.5 + torch.rand(n, branches, per, generator=g)).to(device)
        self.bias = torch.zeros(n, device=device)
        self.fatigue = torch.zeros(n, branches, device=device)
        self.rate = torch.full((n,), k / n, device=device)
        self.age = torch.full((n,), 1e9, device=device)          # the founding population is mature
        self.uses = torch.zeros(n, device=device)
        self.gain = torch.ones(n, device=device)                 # how far each cell is integrated into the code
        self.lr = torch.full((n,), plastic, device=device)       # each cell's own plasticity
        self.lr0 = self.lr.clone()                               # what it was born with
        self.born = self.died = 0
        self.g = torch.Generator(device="cpu").manual_seed(seed + 17)

    @property
    def n(self):
        return self.idx.shape[0]

    def drive(self, x):
        s = (x[:, self.idx] * self.w).sum(-1)                   # (batch, n, branches)
        s = (s - self.fatigue).clamp_min(0.0)
        branch = s * s / (self.theta + s)
        d = branch.sum(-1) - self.bias
        if self.divisive:
            d = d / (1.0 + self.divisive * d.mean(1, keepdim=True).clamp_min(0))
        return d, branch

    @torch.no_grad()
    def code(self, x, learn=False):
        d, branch = self.drive(x)
        # a cell not yet integrated barely competes, and a silent one cannot win at all - even when every mature
        # cell's drive is negative, which multiplying by a gain of zero alone would not prevent
        eff = torch.where(d > 0, d * self.gain, d) - (self.gain <= 0).float() * 1e6
        top = eff.topk(self.k, dim=1).indices
        c = torch.zeros_like(d)
        c.scatter_(1, top, 1.0)
        if learn:
            self._learn(x, c, d, branch)
        return c

    def _learn(self, x, c, d, branch):
        n_ex = x.shape[0]
        self.age += n_ex
        self.uses += c.sum(0)
        if self.homeostasis:
            self.rate.mul_(0.99).add_(0.01 * c.mean(0))
            self.bias.add_(self.homeostasis * (self.rate - self.k / self.n))
        if self.adapt:
            active = c[..., None] * (branch > 0).float()
            self.fatigue.mul_(1 - self.adapt).add_(self.adapt * (active * branch).mean(0))
        # instar Hebbian: a cell that fired moves its synapses toward what it saw. Young cells that are not yet
        # in the code still learn from what drives them most - the "would-have-fired" set among the young -
        # which is how an immature cell becomes selective before it is trusted with a place in the code.
        if self.lr.max() > 0:
            young = self.gain < 1.0
            teach = c.clone()
            if bool(young.any()):
                dy = d[:, young]
                k_y = max(1, min(int(young.sum()), self.k // 4))
                top_y = dy.topk(k_y, dim=1).indices
                ty = torch.zeros_like(dy)
                ty.scatter_(1, top_y, 1.0)
                teach[:, young] = ty
            seen = x[:, self.idx]                                # (batch, n, B, S)
            taught = teach.sum(0)
            # each taught cell moves toward the average of the inputs it was taught on this batch
            upd = (teach[..., None, None] * (seen - self.w[None])).sum(0) / taught.clamp_min(1.0)[:, None, None]
            self.w.add_(self.lr[:, None, None] * upd)
            self.w.clamp_(0.05, 2.0)

    # ------------------------------------------------------------------ growth, maturation, death
    @torch.no_grad()
    def recruit(self, unique_bits, count, young_lr, excitability, integrated):
        """New cells wired to fibres only one member of a confused pair drives (v17's difference wiring)."""
        live = unique_bits.nonzero().squeeze(1)
        if len(live) < 2 or count <= 0:
            return 0
        pick = live[torch.randint(len(live), (count, self.B, self.S), generator=self.g)].to(self.dev)
        stray = torch.randint(self.d_in, (count, self.B, self.S), generator=self.g).to(self.dev)
        mask = (torch.rand(count, self.B, self.S, generator=self.g) < 0.75).to(self.dev)
        self.idx = torch.cat([self.idx, torch.where(mask, pick, stray)])
        self.w = torch.cat([self.w, (0.5 + torch.rand(count, self.B, self.S, generator=self.g)).to(self.dev)])
        self.bias = torch.cat([self.bias, torch.full((count,), -excitability, device=self.dev)])
        self.fatigue = torch.cat([self.fatigue, torch.zeros(count, self.B, device=self.dev)])
        self.rate = torch.cat([self.rate, torch.zeros(count, device=self.dev)])
        self.age = torch.cat([self.age, torch.zeros(count, device=self.dev)])
        self.uses = torch.cat([self.uses, torch.zeros(count, device=self.dev)])
        self.gain = torch.cat([self.gain, torch.full((count,), 1.0 if integrated else 0.0, device=self.dev)])
        self.lr = torch.cat([self.lr, torch.full((count,), young_lr, device=self.dev)])
        self.lr0 = torch.cat([self.lr0, torch.full((count,), young_lr, device=self.dev)])
        self.born += count
        return count

    @torch.no_grad()
    def mature(self, silent, ramp, mature_lr, excitability):
        """Advance every young cell along its maturation: silent to the code at first, then integrated
        gradually; hyperexcitable and highly plastic when born, calming to an ordinary cell by the end."""
        # still maturing, or finished since the last look - a cell that crossed the end of its window between two
        # calls must still be brought all the way in, not left part-integrated forever
        young = (self.age < silent + ramp) | (self.gain < 1.0)
        if not bool(young.any()):
            return 0
        a = self.age[young]
        progress = (a / (silent + ramp)).clamp(0.0, 1.0)
        self.gain[young] = ((a - silent) / max(1.0, ramp)).clamp(0.0, 1.0)
        self.lr[young] = mature_lr + (self.lr0[young] - mature_lr) * (1.0 - progress)
        self.bias[young] = -excitability * (1.0 - progress)
        return int(young.sum())

    @torch.no_grad()
    def cull(self, floor, window, limit, keep_min):
        if self.n <= keep_min:
            return 0
        share = self.k / self.n
        old = self.age > window
        used = self.uses / self.age.clamp(min=1.0)
        doomed = (old & (used < floor * share) & (self.age < 1e8)).nonzero().flatten()
        if not len(doomed):
            return 0
        doomed = doomed[used[doomed].argsort()][:min(limit, self.n - keep_min)]
        keep = torch.ones(self.n, dtype=torch.bool, device=self.dev)
        keep[doomed] = False
        for name in ("idx", "w", "bias", "fatigue", "rate", "age", "uses", "gain", "lr", "lr0"):
            setattr(self, name, getattr(self, name)[keep])
        self.died += int(len(doomed))
        return int(len(doomed))


# ------------------------------------------------------------------------------------------ measuring
@torch.no_grad()
def measure(pop, xa, xb, noise=0.1, seed=0, batch=1024):
    """Over every pair given: separation (1 - overlap of the two codes) and twin recall (a noisy version of each
    sentence is closer to its own clean code than to its twin's). Chance for twin recall is 0.5."""
    g = torch.Generator(device="cpu").manual_seed(seed)
    seps, rec = [], []
    for s in range(0, len(xa), batch):
        a, b = xa[s:s + batch], xb[s:s + batch]
        ca, cb = pop.code(a), pop.code(b)
        seps.append(1.0 - (ca * cb).sum(1) / pop.k)
        for clean, own, other in ((a, ca, cb), (b, cb, ca)):
            flip = (torch.rand(clean.shape, generator=g) < noise).to(clean.device)
            noisy = torch.where(flip & (clean > 0), torch.zeros_like(clean), clean)
            add = (torch.rand(clean.shape, generator=g) < noise * float(clean.mean())).to(clean.device)
            noisy = torch.maximum(noisy, add.float())
            cn = pop.code(noisy)
            rec.append(((cn * own).sum(1) > (cn * other).sum(1)).float()
                       + 0.5 * ((cn * own).sum(1) == (cn * other).sum(1)).float())
    return float(torch.cat(seps).mean()), float(torch.cat(rec).mean())


# ------------------------------------------------------------------------------------------ one arm
ARMS = {
    #              grow   mature  tune   start-size
    "frozen":      (False, False, False, "small"),
    "tune":        (False, False, True,  "small"),
    "grow-instant": (True, False, True,  "small"),    # v17's growth: full strength at birth, culled after 12
    "grow-mature": (True,  True,  True,  "small"),    # the proposal: silent, integrated gradually, protected
    "big-tune":    (False, False, True,  "matched"),  # starts at the size grow-mature ends at
}


def run_arm(arm, seed, cfg, matched_n=None, device="cpu", log=print):
    grow, mature, tune, start = ARMS[arm]
    n0 = cfg["n0"] if start == "small" else int(matched_n)
    pop = Population(cfg["d_in"], n0, cfg["k"], seed=seed, device=device,
                     homeostasis=cfg["homeostasis"] if tune else 0.0, adapt=cfg["adapt"] if tune else 0.0,
                     plastic=cfg["plastic"] if tune else 0.0)
    pairs = stream_pairs(cfg["pairs"], seed)
    xa = encode([a for a, _ in pairs], cfg["d_in"]).to(device)
    xb = encode([b for _, b in pairs], cfg["d_in"]).to(device)
    rng = random.Random(seed)
    blocks = cfg["blocks"]
    per_block = len(pairs) // blocks
    curve, early = [], None
    t0 = time.time()
    for bi in range(blocks):
        lo, hi = bi * per_block, (bi + 1) * per_block
        order = [i for i in range(lo, hi) for _ in range(cfg["repeats"])]
        rng.shuffle(order)
        for s in range(0, len(order), cfg["batch"]):
            ids = torch.tensor(order[s:s + cfg["batch"]], device=device)
            a, b = xa[ids], xb[ids]
            ca = pop.code(a, learn=tune)
            cb = pop.code(b, learn=tune)
            if grow:
                # pressure: how badly each pair in the batch still collides
                overlap = (ca * cb).sum(1) / pop.k
                confused = (overlap > cfg["confused"]).nonzero().flatten()
                budget = cfg["max_units"] - pop.n
                for j in confused.tolist()[: cfg["grow_per_batch"]]:
                    if budget <= 0:
                        break
                    for x_this, x_other in ((a[j], b[j]), (b[j], a[j])):
                        got = pop.recruit(((x_this > 0) & ~(x_other > 0)).float(), cfg["born_per_item"],
                                          young_lr=cfg["young_lr"] if mature else cfg["plastic"],
                                          excitability=cfg["excitability"], integrated=not mature)
                        budget -= got
                if mature:
                    pop.mature(cfg["silent"], cfg["ramp"], cfg["plastic"], cfg["excitability"])
                pop.cull(cfg["floor"], window=(cfg["silent"] + cfg["ramp"]) if mature else 12,
                         limit=cfg["cull_limit"], keep_min=cfg["n0"])
        if bi == 0:
            early = measure(pop, xa[:per_block], xb[:per_block], seed=seed)
        if (bi + 1) % max(1, blocks // 8) == 0 or bi == blocks - 1:
            sep, rec = measure(pop, xa[:hi], xb[:hi], seed=seed)
            curve.append({"block": bi + 1, "pairs_seen": hi, "units": pop.n, "separation": round(sep, 4),
                          "twin_recall": round(rec, 4), "born": pop.born, "died": pop.died,
                          "seconds": round(time.time() - t0)})
            log(f"  [{arm} seed {seed}] {curve[-1]}")
    sep, rec = measure(pop, xa, xb, seed=seed)
    late_early = measure(pop, xa[:per_block], xb[:per_block], seed=seed)
    return {"arm": arm, "seed": seed, "units": pop.n, "born": pop.born, "died": pop.died,
            "separation": sep, "twin_recall": rec, "curve": curve,
            "first_block_then": early, "first_block_end": late_early,
            "forgetting_separation": round(early[0] - late_early[0], 4),
            "forgetting_recall": round(early[1] - late_early[1], 4), "seconds": round(time.time() - t0)}


DEFAULT = {"d_in": 4096, "n0": 1024, "k": 32, "pairs": 4000, "blocks": 20, "repeats": 5, "batch": 64,
           "homeostasis": 0.002, "adapt": 0.02, "plastic": 0.01, "young_lr": 0.1, "excitability": 0.3,
           "silent": 2000, "ramp": 4000, "confused": 0.5, "grow_per_batch": 4, "born_per_item": 4,
           "max_units": 4096, "floor": 0.2, "cull_limit": 32}


def analyse(results, alpha=0.05):
    from scipy import stats
    import numpy as np
    by = {}
    for r in results:
        by.setdefault(r["arm"], {})[r["seed"]] = r

    def paired(a, b, key):
        seeds = sorted(set(by.get(a, {})) & set(by.get(b, {})))
        if len(seeds) < 2:
            return None
        x = np.array([by[a][s][key] for s in seeds])
        y = np.array([by[b][s][key] for s in seeds])
        t, p = stats.ttest_rel(x, y)
        return float((x - y).mean()), float(t), float(p)

    tests = []
    for name, a, b, key in [("H1 grow-mature > tune (separation, all pairs)", "grow-mature", "tune", "separation"),
                            ("H2 grow-mature > big-tune (separation)", "grow-mature", "big-tune", "separation"),
                            ("H3 grow-mature > grow-instant (separation)", "grow-mature", "grow-instant",
                             "separation"),
                            ("H4 grow-mature > tune (twin recall)", "grow-mature", "tune", "twin_recall")]:
        got = paired(a, b, key)
        if got:
            tests.append((name,) + got)
    order = sorted(range(len(tests)), key=lambda i: tests[i][3])
    verdict, alive = {}, True
    for rank, i in enumerate(order):
        name, diff, t, p = tests[i]
        alive = alive and p <= alpha / (len(tests) - rank) and diff > 0
        verdict[name] = "SUPPORTED" if alive else "not supported"
    means = {arm: {k: round(float(np.mean([r[k] for r in rs.values()])), 4)
                   for k in ("separation", "twin_recall", "units", "forgetting_separation", "forgetting_recall")}
             for arm, rs in by.items()}
    return {"tests": [dict(zip(("name", "diff", "t", "p"), x)) for x in tests], "verdict": verdict, "means": means}


def main():
    import argparse
    import os
    ap = argparse.ArgumentParser()
    ap.add_argument("--seeds", type=int, default=5)
    ap.add_argument("--out", default="results/maturation")
    ap.add_argument("--pairs", type=int, default=DEFAULT["pairs"])
    ap.add_argument("--pilot", action="store_true", help="a small, fast run of every arm, before the real one")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    cfg = dict(DEFAULT, pairs=a.pairs)
    if a.pilot:
        cfg.update(pairs=400, blocks=4, silent=200, ramp=400)
    print("device", dev, "config", json.dumps(cfg), flush=True)
    results = []
    for seed in range(a.seeds if not a.pilot else 1):
        grown = None
        for arm in ("frozen", "tune", "grow-instant", "grow-mature", "big-tune"):
            r = run_arm(arm, seed, cfg, matched_n=grown, device=dev)
            if arm == "grow-mature":
                grown = r["units"]
            results.append(r)
            print(f"== {arm} seed {seed}: separation {r['separation']:.4f} twin recall {r['twin_recall']:.4f} "
                  f"units {r['units']} (born {r['born']}, died {r['died']}) forgetting "
                  f"{r['forgetting_separation']:+.4f} | {r['seconds']}s", flush=True)
            json.dump({"config": cfg, "results": results}, open(os.path.join(a.out, "results.json"), "w"), indent=1)
    if not a.pilot:
        verdict = analyse(results)
        json.dump({"config": cfg, "results": results, "analysis": verdict},
                  open(os.path.join(a.out, "results.json"), "w"), indent=1)
        print(json.dumps(verdict, indent=1), flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/brain/tests/check_maturation.py
"""The maturation experiment's machinery, checked before anything is concluded from it (runs on Kaggle)."""
import os
import sys

import torch

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, os.path.dirname(HERE))

from brain.maturation import Population, encode, measure, stream_pairs, names  # noqa: E402

FAILED = []


def check(name, cond, detail=""):
    print(f"{'PASS' if cond else 'FAIL'}  {name}" + (f"   {detail}" if detail and not cond else ""))
    if not cond:
        FAILED.append(name)


# the sentences
pairs = stream_pairs(200, seed=0)
check("every pair is two different sentences", all(a != b for a, b in pairs))
twins = names(50, 0)
check("near-twin names differ by one letter", all(sum(x != y for x, y in zip(a, b)) == 1 for a, b in twins))
x = encode(["Rex is older than Max", "Max is older than Rex"])
words = encode(["Rex", "is", "older", "than", "Max"])
check("a role swap shares its word fibres", bool(((x[0] > 0) & (x[1] > 0)).sum() >= 24))
check("and differs in its position-bound fibres", bool(((x[0] > 0) ^ (x[1] > 0)).sum() > 0))

# the population
pop = Population(4096, 256, 16, seed=0, homeostasis=0.002, adapt=0.02, plastic=0.01)
xs = encode([a for a, _ in pairs[:64]])
c = pop.code(xs, learn=True)
check("exactly k cells fire", bool((c.sum(1) == 16).all()), str(c.sum(1)[:5]))
pop.recruit((xs[0] > 0).float(), 8, young_lr=0.1, excitability=0.3, integrated=False)
check("newborns are counted", pop.n == 264 and pop.born == 8)
c = pop.code(xs)
check("a silent newborn never enters the code", float(c[:, 256:].sum()) == 0.0)
pop.age[256:] = 300.0
pop.mature(silent=200, ramp=400, mature_lr=0.01, excitability=0.3)
g = float(pop.gain[256])
check("half-way through the ramp a newborn is partly integrated", 0.1 < g < 0.4, f"{g}")
check("and its plasticity has fallen toward the adult level", 0.01 < float(pop.lr[256]) < 0.1)
pop.age[256:] = 700.0
pop.mature(silent=200, ramp=400, mature_lr=0.01, excitability=0.3)
check("after maturing it is fully integrated", float(pop.gain[256]) == 1.0)
before = pop.n
pop.uses.zero_()
pop.age[:256] = 1e9
pop.age[256:260] = 100.0
died = pop.cull(floor=0.2, window=600, limit=100, keep_min=16)
check("culling never touches the founders or the still-young", died == 4 and pop.n == before - 4, f"{died}")

# the measure
same = measure(pop, xs[:32], xs[:32])
check("identical sentences have zero separation", abs(same[0]) < 1e-6, f"{same}")
diff = measure(pop, encode([a for a, _ in pairs[:32]]), encode([b for _, b in pairs[:32]]))
check("near-identical sentences have some", 0.0 < diff[0] < 1.0, f"{diff}")
check("twin recall is a proportion", 0.0 <= diff[1] <= 1.0)

print("\n" + ("ALL PASS" if not FAILED else f"{len(FAILED)} FAILED: " + ", ".join(FAILED)))
sys.exit(1 if FAILED else 0)

In [ ]:
%%writefile /kaggle/working/brain/PREREGISTRATION_maturation.md
# Pre-registration: does cortical growth help once newborn cells mature and the task outgrows the population?

Written 2026-09-24, before any run of `brain/maturation.py` (no local runs at all; the Kaggle notebook runs the
checks, then a one-seed pilot, then this experiment). Code at the commit this file is in.

## Why

v17 (PREREGISTRATION_development.md): critical-period growth lost to tuning alone (separation 0.760 vs 0.808,
critical below tune on all 5 seeds). The user's objection: newborn cells were never allowed to mature, and the
tasks were too simple for growth to matter. Both are fair - v17 newborns joined the code at full strength at
birth and could be culled after 12 exposures, and tuning alone already separated v17's curriculum, so the
population never ran out of room.

## Design

A population of 1,024 cells (k = 32 active, 4 dendritic branches x 6 synapses, supralinear branches, divisive
gain control) meets a continual stream of 4,000 near-identical sentence pairs (role swaps, one-letter name
twins, negation, tense, one-letter places, direction, scale) in 20 blocks, each pair presented 5 times within its
block and never again. Inputs are position-bound word codes (4,096 fibres). That is 8,000 sentences to keep
apart with 1,024 cells: capacity runs out.

Arms (5 seeds; the seed sets the sentences, the wiring and the order):

| arm | growth | newborns | tuning (homeostasis + adaptation + instar plasticity) | size |
|---|---|---|---|---|
| frozen | no | - | no | 1,024 |
| tune | no | - | yes | 1,024 |
| grow-instant | yes | full strength at birth, cullable after 12 exposures (v17) | yes | grows |
| grow-mature | yes | silent 2,000 exposures, integrated over 4,000 more; 10x plasticity decaying to adult; hyperexcitable decaying; cullable only once mature | yes | grows |
| big-tune | no | - | yes | the final size of grow-mature, same seed |

Growth is driven by pressure: a pair whose two codes still overlap by more than half recruits 4 cells for each
member, wired mostly to the fibres only that member drives; at most 4,096 cells. Death: a mature newborn used
at under 20% of its share is removed (the founders are never culled).

Measures, on all 4,000 pairs at the end: **separation** (1 - overlap of the two codes, mean over pairs) and
**twin recall** (a sentence with 10% of its fibres flipped comes back closer to its own clean code than to its
twin's; chance 0.5). Also forgetting: the first block's separation right after it was learned minus at the end.

## Hypotheses

- **H1 (primary).** grow-mature > tune on separation.
- **H2.** grow-mature > big-tune on separation. (Growing beats simply starting that big.)
- **H3.** grow-mature > grow-instant on separation. (Maturation is what makes growth work.)
- **H4.** grow-mature > tune on twin recall.

Paired t across 5 seeds (df 4), two-sided, alpha 0.05, Holm-Bonferroni across H1-H4.

## How it will be read

- H1 and H2 supported: growth with maturation earns its place, and not merely by adding cells.
- H1 supported, H2 not: more cells help; growing them is not what helps.
- H3 supported: the v17 failure was the missing maturation, as argued.
- H1 fails: under capacity pressure and with maturation, cortical growth still does not help in this model.

The pilot inside the same Kaggle run (one seed, 400 pairs, shorter maturation) is only a check that every arm
runs; its numbers are not used to change anything above.

### Checks

In [ ]:
p = subprocess.Popen([sys.executable, "-u", *['tests/check_maturation.py']], cwd=W, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="", flush=True)
assert p.wait() == 0, "failed"

### Pilot: one seed, small, every arm (not used to change anything)

In [ ]:
p = subprocess.Popen([sys.executable, "-u", *['brain/maturation.py', '--pilot', '--out', '/kaggle/working/pilot']], cwd=W, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="", flush=True)
assert p.wait() == 0, "failed"

### The pre-registered experiment

In [ ]:
p = subprocess.Popen([sys.executable, "-u", *['brain/maturation.py', '--seeds', '5', '--out', '/kaggle/working/maturation']], cwd=W, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="", flush=True)
assert p.wait() == 0, "failed"